# 📓 Lab 05 — Attention y bloque Transformer desde cero

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 3 · Laboratorio 3**
**Material del aula:** [Sesión 3](../sesiones/03-secuencias-transformers.md) ·
**Implementación de referencia:** [`src/models.py`](../src/models.py) ·
**Pruebas:** [`tests/test_shapes.py`](../tests/test_shapes.py)

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V$$

($M$ = máscara: 0 en posiciones permitidas, $-\infty$ en las prohibidas — se construye en la sección 3.)

Plan: (1) una fila de attention A MANO, (2) la función completa con
validaciones, (3) máscara causal + pruebas, (4) multi-head + bloque
Transformer, (5) heatmaps.

🕹️ Ten abierto el [simulador de attention](https://felmco.github.io/deeplearning-class/interactivos/atencion.html)
para verificar cada paso.

In [ ]:
from __future__ import annotations

import math
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import torch

from src.utils import seed_everything
from src.models import (scaled_dot_product_attention,
                        MultiHeadSelfAttention, TransformerBlock,
                        positional_encoding_sinusoidal)

seed_everything(42)

## 1. Una fila de attention, a mano

La actividad manual de la sesión, ahora verificada con código.
Tres tokens, $d_k = 2$:

$$Q=K=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\qquad
V=\begin{bmatrix}1&2\\3&0\\0&4\end{bmatrix}$$

In [ ]:
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = Q.clone()
V = torch.tensor([[1., 2.], [3., 0.], [0., 4.]])

# Paso 1: compatibilidades query-key
scores = Q @ K.T
print('scores = QKᵀ:'); print(scores)

# Paso 2: escalar por √d_k (aquí √2)
escalado = scores / math.sqrt(2)
print('\nscores/√2:'); print(escalado)

# Paso 3: softmax de la PRIMERA fila (a mano)
fila = escalado[0]
softmax_fila = torch.exp(fila - fila.max()) / torch.exp(fila - fila.max()).sum()
print('\nsoftmax fila 0:', softmax_fila, '→ suma =', softmax_fila.sum().item())

# Paso 4: combinación ponderada de V para el token 0
salida_token0 = softmax_fila @ V
print('salida token 0:', salida_token0)

# ✏️ Compara estos números con tu cálculo manual de la sesión.

## 2. La función completa (implementación de referencia)

`scaled_dot_product_attention` vive en [`src/models.py`](../src/models.py),
comentada paso a paso: contrato de shapes, escalado, máscara con `-inf`,
softmax por filas y mezcla de values. Léela AHORA antes de continuar.

In [ ]:
# B=1 batch, H=1 head, T=4 tokens, d_k=d_v=4
seed_everything(42)
q = torch.randn(1, 1, 4, 4, requires_grad=True)
k = torch.randn(1, 1, 4, 4, requires_grad=True)
v = torch.randn(1, 1, 4, 4, requires_grad=True)

salida, pesos = scaled_dot_product_attention(q, k, v)
print('salida:', salida.shape)    # (1, 1, 4, 4)
print('pesos :', pesos.shape)     # (1, 1, 4, 4) — matriz token × token

# Propiedad fundamental: cada fila es una distribución (suma 1)
print('sumas por fila:', pesos.sum(dim=-1).squeeze())

## 3. Máscara causal + las 4 pruebas del laboratorio

La máscara triangular impide mirar al futuro. Las pruebas siguientes son
las MISMAS de `tests/test_shapes.py` — aquí las ejecutas visiblemente
(en el repo corren con `pytest`).

In [ ]:
causal = torch.tril(torch.ones(4, 4, dtype=torch.bool)).view(1, 1, 4, 4)
salida, pesos = scaled_dot_product_attention(q, k, v, mask=causal)

# ── Prueba 1: shapes correctos ──
assert salida.shape == (1, 1, 4, 4)
assert pesos.shape == (1, 1, 4, 4)

# ── Prueba 2: cada fila suma 1 ──
assert torch.allclose(pesos.sum(dim=-1), torch.ones(1, 1, 4), atol=1e-6)

# ── Prueba 3: el futuro está bloqueado ──
assert torch.all(pesos.masked_select(~causal) < 1e-6)

# ── Prueba 4: backpropagation funciona ──
loss = salida.square().mean()
loss.backward()
assert q.grad is not None and torch.isfinite(q.grad).all()

print('✅ Las 4 pruebas pasan')

In [ ]:
# Heatmap de la attention causal: el triángulo superior es CERO
plt.figure(figsize=(5, 4))
plt.imshow(pesos.detach().squeeze().numpy(), cmap='viridis')
plt.xlabel('Key (posición atendida)')
plt.ylabel('Query (posición que consulta)')
plt.title('Causal attention weights')
plt.colorbar()
plt.show()

### Comparación con la implementación oficial de PyTorch

Nuestra versión pedagógica debe coincidir numéricamente con
`F.scaled_dot_product_attention` (que usa kernels optimizados).

In [ ]:
referencia = torch.nn.functional.scaled_dot_product_attention(
    q.detach(), k.detach(), v.detach(), attn_mask=causal
)
diferencia = torch.max(torch.abs(referencia - salida.detach()))
print(f'diferencia máxima vs PyTorch: {diferencia:.2e}')
# Nota: máscaras/kernels pueden variar por versión y dispositivo;
# validar shapes y proximidad numérica en TU entorno.

## 4. Multi-head attention y el bloque Transformer

[`src/models.py`](../src/models.py) implementa:
- `MultiHeadSelfAttention`: h proyecciones paralelas + concat + $W^O$;
(MHA = multi-head attention, LN = LayerNorm, FFN = red feed-forward de 2 capas lineales.)

- `TransformerBlock` pre-norm: $x' = x + \text{MHA}(\text{LN}(x))$,
  $y = x' + \text{FFN}(\text{LN}(x'))$.

Probamos el bloque completo con máscara causal Y padding mask a la vez.

In [ ]:
seed_everything(42)
x = torch.randn(2, 7, 64, requires_grad=True)   # batch 2, 7 tokens, d_model 64

# La secuencia 0 tiene solo 5 tokens reales (2 de padding)
padding_mask = torch.tensor([
    [True, True, True, True, True, False, False],
    [True, True, True, True, True, True,  True],
])

block = TransformerBlock(d_model=64, num_heads=4, d_ff=128)  # d_ff: dimensión interna de la FFN
y, pesos_attn = block(x, causal=True, padding_mask=padding_mask)

assert y.shape == x.shape                      # el bloque PRESERVA la shape
assert pesos_attn.shape == (2, 4, 7, 7)        # (B, heads, T, T)

y.mean().backward()                            # gradientes de punta a punta
assert x.grad is not None
print('✅ Bloque Transformer: shapes y backprop OK')
print('salida:', y.shape, '| attention:', pesos_attn.shape)

In [ ]:
# Heatmaps de las 4 heads de la secuencia 0:
# cada head aprende (tras entrenar) un patrón de ruteo distinto
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for h, ax in enumerate(axes):
    im = ax.imshow(pesos_attn[0, h].detach().numpy(), cmap='viridis')
    ax.set_title(f'head {h}')
    ax.set_xlabel('key'); ax.set_ylabel('query' if h == 0 else '')
fig.suptitle('Multi-head: 4 miradas paralelas (sin entrenar — patrones aleatorios)',
             fontweight='bold')
plt.tight_layout(); plt.show()

# ⚠️ Reflexión: estos mapas muestran RUTEO de información.
# Un peso alto NO demuestra una relación lingüística causal.

## 5. Positional encoding

La self-attention es permutacionalmente invariante: sin señal de posición,
"perro muerde hombre" = "hombre muerde perro".

In [ ]:
pe = positional_encoding_sinusoidal(max_len=100, d_model=64)

plt.figure(figsize=(9, 4.5))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.xlabel('dimensión del embedding')
plt.ylabel('posición en la secuencia')
plt.title('Positional encoding sinusoidal: una firma única por posición')
plt.colorbar()
plt.show()

# Demostración de la invariancia: permutar la entrada permuta la salida
seed_everything(1)
mha = MultiHeadSelfAttention(d_model=64, num_heads=4)
mha.eval()
x1 = torch.randn(1, 5, 64)
perm = torch.tensor([4, 2, 0, 3, 1])
with torch.inference_mode():
    salida_normal, _ = mha(x1)
    salida_perm, _ = mha(x1[:, perm])
print('¿salida permutada == permutación de la salida?',
      torch.allclose(salida_perm, salida_normal[:, perm], atol=1e-5))

## 6. 🔬 Cierre con Transformer Explainer

Abre <https://poloclub.github.io/transformer-explainer/> y sigue el
[guion de 10 pasos de la Sesión 3](../sesiones/03-secuencias-transformers.md#7--laboratorio-visual--transformer-explainer).
Compara **dos prompts** y registra: tokenización, una head interesante,
efecto de la temperatura y de top-k/top-p.

## 📝 Entrega del Laboratorio 3

- [ ] Implementación con validaciones de shapes (o lectura crítica de `src/models.py`)
- [ ] Prueba: cada fila de pesos suma ≈ 1
- [ ] Prueba: la máscara causal bloquea el futuro
- [ ] Backpropagation exitoso a través del bloque
- [ ] Heatmap de una head
- [ ] Comparación de dos prompts en Transformer Explainer
- [ ] Reflexión: ¿qué puede inferirse y qué NO de un attention map?
- [ ] `git commit -m "feat: complete attention lab"`